In [1]:
import pandas as pd
from data.loaders import load_single_target_tba, load_gsk_hepg2, load_pk, load_derbyshire_hepg2, load_derbyshire_malaria, load_dual_target_tba
from evaluate.train import generate_repeated_5xn_splits, calc_metrics
from config import SplitType, TrainConfig
from data.preprocessing import preprocess_ray
import ray

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
df, df_classification_threshold = load_single_target_tba()
df = preprocess_ray(df, use_features=False, drop_nan_features=True)
ray.shutdown()

/home/rahul/delta/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:54: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
2026-05-11 10:04:20,716	INFO worker.py:1998 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/home/rahul/delta/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-05-11 10:04:27,792	INFO logging.py:397 -- Registered dataset logger for dataset dataset_1_0
2026-05-11 10:04:27,828	INFO streaming_executor.py:178 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-05-11_10-04-17_351642_8871/logs/ray-data
202

In [4]:
train_config = TrainConfig(batch_size=16, split_type=SplitType.SCAFFOLD, max_epochs=25)
splits = generate_repeated_5xn_splits(
    df,
    train_config.n_splits,
    train_config.split_type,
    random_state=train_config.random_seed,
)

_ = next(splits)
_, (train_df, val_df, test_df) = next(splits)

/home/rahul/delta/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(
/home/rahul/delta/.venv/lib/python3.12/site-packages/sklearn/model_selection/_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


In [5]:
from models.deltaprop import DeltapropRef
from models.config import DeltapropConfig

train_split, val_split, test_split, extras = DeltapropRef.prepare_splits(
    train_df=train_df, val_df=val_df, test_df=test_df,
)

model_config = DeltapropConfig(encoder_n_layers=3, candidate_size=16, use_chameleon_mp=True)
model = DeltapropRef.build(model_config=model_config, **extras)

In [6]:
model = model.train_func(
    train_split=train_split,
    val_split=val_split,
    train_config=train_config,
    model_config=model_config,
    df_classification_threshold=df_classification_threshold,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/rahul/delta/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (26) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing │  8.7 M │ train │     0 │
│ 1 │ agg             │ NormAggregation    │      0 │ train │     0 │
│ 2 │ encoder         │ Encoder            │  705 K │ train │     0 │
│ 3 │ interaction     │ Interaction        │ 90.6 K │ train │     0 │
│ 4 │ bn              │ Identity           │      0 │ train │     0 │
│ 5 │ X_d_transform   │ Identity           │      0 │ train │     0 │
│ 6 │ loss_fn         │ BCEWithLogitsLoss  │      0 │ train │     0 │
└───┴─────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 796 K                                                                                            
Non-trainable params: 8.7 M                                                                                        
Total params: 9.5 M                                                                                                
Total estimated model params size (MB): 38                                                                         
Modules in train mode: 28                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 0.624


Metric val_loss improved by 0.070 >= min_delta = 0.0. New best score: 0.554


Metric val_loss improved by 0.057 >= min_delta = 0.0. New best score: 0.497


Monitored metric val_loss did not improve in the last 20 records. Best score: 0.497. Signaling Trainer to stop.
`Trainer.fit` stopped: `max_epochs=25` reached.


In [7]:
clf_th = model.tune_binary_classification_threshold(
    val_split=val_split,
    val_labels=val_df["bin_target"],
    train_config=train_config,
    train_split=train_split,
    train_labels=train_df["bin_target"],
    df_classification_threshold=df_classification_threshold,
)

pred_probs, preds = model.predict_func(
    binary_classification_threshold=clf_th,
    train_split=train_split,
    train_labels=train_df["bin_target"],
    test_split=test_split,
    df_classification_threshold=df_classification_threshold,
)

In [8]:
calc_metrics(pred_probs, preds, test_df["bin_target"])

{'accuracy': 0.75,
 'balanced_accuracy': 0.7548148148148148,
 'f1': 0.7719298245614035,
 'precision': 0.6875,
 'recall': 0.88,
 'mcc': 0.5233793327271588,
 'roc_auc': 0.8503703703703703,
 'average_precision': 0.8681822921050009}

In [10]:
calc_metrics(pred_probs, preds, test_df["bin_target"])

{'accuracy': 0.6730769230769231,
 'balanced_accuracy': 0.6837037037037037,
 'f1': 0.7384615384615385,
 'precision': 0.6,
 'recall': 0.96,
 'mcc': 0.43569158873431,
 'roc_auc': 0.84,
 'average_precision': 0.8455180978016018}

In [15]:
calc_metrics(pred_probs, preds, test_df["bin_target"])

{'accuracy': 0.5384615384615384,
 'balanced_accuracy': 0.5555555555555556,
 'f1': 0.6756756756756757,
 'precision': 0.5102040816326531,
 'recall': 1.0,
 'mcc': 0.23809523809523808,
 'roc_auc': 0.8622222222222222,
 'average_precision': 0.8793044214769212}

In [ ]:
from models.deltaprop.data import build_discordancy_matrix
import numpy as np

from models.deltaprop.utils import score_all

theta_hat_train = score_all(train_split, model.model).squeeze().cpu().numpy()
reference_scores = train_df['cont_target'].to_numpy()
nu = np.exp(model.model.interaction.log_nu.cpu().item())
D = build_discordancy_matrix(theta_hat_train, reference_scores, nu)

In [ ]:
asd = (D > 0).sum(axis=1)

In [ ]:
sidxs = asd.argsort()

In [ ]:
train_df['bin_target'][sidxs].to_numpy()[:100]

In [ ]:
asd[sidxs]

In [ ]:
calc_metrics(pred_probs, preds, test_df["bin_target"])

In [ ]:
# no hards
calc_metrics(pred_probs, preds, test_df["bin_target"])

In [ ]:
from models.deltaprop.data import RandomPairDataModule


datamodule = RandomPairDataModule(
    train_mol_ds=train_split, 
    val_mol_ds=val_split,
    binary_threshold=df_classification_threshold,
    batch_size=train_config.batch_size,
    n_candidates=model_config.candidate_size,
)

In [ ]:
import torch

from models.deltaprop.model import embed_all


train_mol_ds = datamodule.train_ds.anchor_dataset
train_embeds = embed_all(train_mol_ds, model.model)

with torch.no_grad():
    theta_hat_train = (
        model.model.interaction.projector(train_embeds).squeeze()
    )

    pred_probs = model.model.interaction._davidson_logit(
        theta_hat_train.unsqueeze(0), theta_hat_train.unsqueeze(1), model.model.interaction.log_nu
    ).sigmoid().squeeze().cpu()

    reference = torch.from_numpy(train_mol_ds.Y >= train_mol_ds.Y.T)
    negatives = torch.logical_xor(pred_probs > 0.5, reference)

    abs_diffs = torch.from_numpy(train_mol_ds.Y - train_mol_ds.Y.T).abs()
    negative_idxs_sorted = torch.argsort(pred_probs * abs_diffs, descending=True)

    # hard_neg_idxs = []
    # for i in range(negatives.shape[0]):
    #     idxs = torch.argwhere(negatives[i]).squeeze()
    #     if isinstance(idxs, int):
    #         idxs = [idxs]
        
    #     sorted_idxs = torch.argsort(degree_negativity[i, idxs], descending=True).tolist()
    #     hard_neg_idxs.append(sorted_idxs)
        # print(i, idxs)

    # print(hard_neg_idxs[0])
    # datamodule.train_ds.hard_neg_idxs = hard_neg_idxs

In [ ]:
asd = pred_probs * abs_diffs

In [ ]:
asd_masked = torch.where(negatives, asd, float("-inf"))

In [ ]:
non_zeros = torch.argwhere(asd_masked[0] > 0)

In [ ]:
torch.nn.functional.softmax(asd_masked, dim=-1)

In [ ]:
asd_masked.max(dim=-1)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.hist(torch.nn.functional.softmax(asd_masked, dim=-1)[1].numpy())

In [ ]:
from models.deltaprop import embed_all

In [ ]:
train_embeds = embed_all(train_split, model.model)
val_embeds = embed_all(val_split, model.model)
test_embeds = embed_all(test_split, model.model, scale_X_d=True)

In [ ]:
import torch

with torch.no_grad():
    theta_hat_train = (
        model.model.interaction.projector(train_embeds).squeeze().cpu()
    )

    theta_hat_val = (
        model.model.interaction.projector(val_embeds).squeeze().cpu()
    )

    theta_hat_test = (
        model.model.interaction.projector(test_embeds).squeeze().cpu()
    )

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

In [ ]:
plt.scatter(
    train_split.Y.squeeze(),
    theta_hat_train,
    c=train_df['bin_target']
)

spearmanr(
    train_split.Y.squeeze(),
    theta_hat_train
)

In [ ]:
plt.scatter(
    val_split.Y.squeeze(),
    theta_hat_val,
    c=val_df['bin_target']
)

spearmanr(
    val_split.Y.squeeze(),
    theta_hat_val
)

In [ ]:
plt.scatter(
    test_split.Y.squeeze(),
    theta_hat_test,
    c=test_df['bin_target']
)

spearmanr(
    test_split.Y.squeeze(),
    theta_hat_test
)

In [ ]:
import numpy as np

In [ ]:
reference = np.argsort(train_split.Y.squeeze())
actual = np.argsort(theta_hat_train)

In [ ]:
def discordant_for(i, reference, predicted):
    pred_order = predicted[i] - predicted   # compare i against all j
    ref_order  = reference[i]  - reference
    
    discordant_mask = pred_order * ref_order < 0
    discordant_mask[i] = False              # exclude self
    
    return np.where(discordant_mask)[0]

In [ ]:
discordant_for(5, train_split.Y.squeeze(), theta_hat_train.numpy())

In [ ]:
reference

In [ ]:
actual

In [ ]:
from sklearn.isotonic import IsotonicRegression


iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
iso.fit(theta_hat_train, train_split.Y.squeeze())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for z in range(0, pred_probs.shape[0], 50):

    

    pos_mask = train_df['cont_target'] > df_classification_threshold.th
    neg_mask = ~pos_mask
    pos_prob = pred_probs[z][pos_mask].mean()
    neg_prob = pred_probs[z][neg_mask].mean()
    alt_prob = round(float((pos_prob + neg_prob) / 2), 2)

    # if not (~test_df['bin_target'].iloc[z] and alt_prob > 0.5):
    #     continue

    Y = pred_probs[z]
    X = train_df['cont_target'].to_numpy()

    plt.scatter(X, Y)
    # for i, label in enumerate(sims[z][topk_train_cand_idxs]):
    #     plt.text(X[i], Y[i], round(label, 3), fontsize=12)
    
    plt.axvline(test_df['cont_target'].iloc[z], color="blue")
    plt.axvline(df_classification_threshold.th, color="red")
    plt.axhline(0.5, color="red")
    # plt.ylim(0, 1)

    # pos_mask = train_cand_df['cont_target'] > df_classification_threshold.th
    # neg_mask = ~pos_mask
    # pos_prob = np.nan_to_num(Y[pos_mask].mean())
    # neg_prob = np.nan_to_num(Y[neg_mask].mean())
    # prob = round(float((pos_prob + neg_prob) / 2), 2)

    # pos_mask = train_cand_df['cont_target'] > df_classification_threshold.th
    # neg_mask = ~pos_mask
    # pos_prob = (Y[pos_mask] * X[pos_mask]).mean()
    # neg_prob = (Y[neg_mask] * X[neg_mask]).mean()
    # alt_prob = round(float((pos_prob + neg_prob) / 2), 2)



    

    plt.title(f"{test_df['cont_target'].iloc[z]}, {alt_prob}, {pos_prob}")
    plt.show()
    # break

In [ ]:
# from sklearn.metrics import average_precision_score, roc_auc_score

# scores = []
# for idx in range(train_embeds.shape[0]):
#     scores.append(
#         roc_auc_score(test_df['bin_target'], pred_probs[:, idx])
#     )

# scores = np.array(scores)
# sorted_idxs = np.argsort(scores)[::-1]
# list(zip(train_df['cont_target'].iloc[sorted_idxs].astype(float), scores[sorted_idxs]))

In [ ]:
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import numpy as np


def tanimoto_similarity_matrix(
    mols_a: list,
    mols_b: list,
    radius: int = 2,
    n_bits: int = 2048,
) -> np.ndarray:
    """
    Calculate Tanimoto similarity between two lists of RDKit molecules.

    Args:
        mols_a: First list of RDKit Mol objects.
        mols_b: Second list of RDKit Mol objects.
        radius: Morgan fingerprint radius (default: 2).
        n_bits: Number of bits in the fingerprint (default: 2048).

    Returns:
        A (len(mols_a), len(mols_b)) numpy array of Tanimoto similarities.
    """
    generator = GetMorganGenerator(radius=radius, fpSize=n_bits)

    def to_fp(mol):
        if mol is None:
            raise ValueError("Invalid molecule (None) in input list.")
        return generator.GetFingerprint(mol)

    fps_a = [to_fp(m) for m in mols_a]
    fps_b = [to_fp(m) for m in mols_b]

    similarity_matrix = np.zeros((len(fps_a), len(fps_b)))

    for i, fp_a in enumerate(fps_a):
        similarity_matrix[i] = DataStructs.BulkTanimotoSimilarity(fp_a, fps_b)

    return similarity_matrix

In [ ]:
sims = tanimoto_similarity_matrix(test_mol_ds.mols, train_mol_ds.mols)
sims_topk = np.argsort(sims, axis=-1)[:, ::-1][:, :10]

sims_topk_mask = np.zeros_like(sims, dtype=np.bool)
np.put_along_axis(sims_topk_mask, sims_topk, True, axis=-1)

In [ ]:
import torch

with torch.no_grad():
    pred_probs = (
        model.interaction(test_embeds, train_embeds)
        .sigmoid()
        .squeeze()
        .cpu()
        .numpy()
    )

In [ ]:
df_classification_threshold

In [ ]:
pos_mask = train_df["bin_target"].to_numpy()
neg_mask = ~pos_mask

In [ ]:
# pos_contrib = pred_probs[:, sims_topk_mask]
# neg_contrib = pred_probs[:, sims_topk_mask]

np.where(pos_mask & sims_topk_mask, pred_probs, 0.0)

In [ ]:
pos_contrib_mask = pos_mask & sims_topk_mask
neg_contrib_mask = neg_mask & sims_topk_mask

In [ ]:
(pred_probs * pos_contrib_mask).sum(axis=-1) / (pos_contrib_mask.sum(axis=-1) + 1e-12)

In [ ]:
pos_contrib = (pred_probs * pos_contrib_mask).sum(axis=-1) / (pos_contrib_mask.sum(axis=-1) + 1e-12)
neg_contrib = (pred_probs * neg_contrib_mask).sum(axis=-1) / (neg_contrib_mask.sum(axis=-1) + 1e-12)

pos_prob = np.nan_to_num(pos_contrib)
neg_prob = np.nan_to_num(neg_contrib)
prob = (pos_prob + neg_prob) / 2

In [ ]:
train_df['cont_target'].hist()